# T2.5 Analyze the models with respect to different groups


In [15]:
from __future__ import annotations

import re
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.patches import Patch

from scipy import stats
from pathlib import Path
from collections.abc import Callable

from source.metrics.util import RecommenderDataLoader
from source.metrics.beyond_accuracy import BeyondAccuracyMetrics
from source.metrics.ranking_metrics import RankingMetrics

from dataclasses import dataclass
from itertools import groupby, product

In [25]:
# Config
MODELS = list(dict.fromkeys(["BPR", "EASE", "FISM", "ItemKNN", "UserKNN", "SLIMElastic", "NeuMF", "NeuMF", "LightGCN", "NGCF", "Pop", "Random", "TunedAugmentation", "TunedCascade", "TunedCombination", "TunedMetalevel", "TunedMixed", "TunedSwitching", "TunedWeighted"]))
MODEL_COLORS = dict(zip(MODELS, plt.cm.tab20.colors))  # one colour per model, used in every plot
K = 10
ALPHA = 0.1  # FDR level used when flagging cells

HIDDEN_SPLITS = ("train", "validation")  # never re-recommended
TEST_SPLIT = "test"  # "unassigned" pairs remain candidates

ACTIVITY_LABELS = ["low", "mid-low", "mid-high", "high"]
POPULARITY_LABELS = ["long-tail", "low-mid", "high-mid", "head"]
USER_DIMS = ["age_group", "gender", "occupation", "activity"]

In [3]:
def load_evaluation_data(prediction_file: str):
    """
    Helper funtion to load scores, align IDs, and build boolean matrices.
    Returns: (user_id2idx, item_id2idx, predicted_scores, train_mask, test_mask, validation_mask)
    """
    with RecommenderDataLoader(base_dir="../../results") as loader:
        users, items, predicted_scores = loader.load_predictions(npz_filename=prediction_file, npz_dir="tuned_predictions")
        num_users, num_items = predicted_scores.shape

        user_id2idx = {int(raw_id): idx for idx, raw_id in enumerate(users)}
        item_id2idx = {int(raw_id): idx for idx, raw_id in enumerate(items)}

        def build_dense_mask(filename: str) -> np.ndarray:
            mask = np.zeros((num_users, num_items), dtype=bool)
            try:
                pairs = loader.load_split_data_file(filename)
                for row in pairs:
                    if len(row) >= 2:
                        raw_u, raw_i = int(row[0]), int(row[1])
                        if raw_u in user_id2idx and raw_i in item_id2idx:
                            mask[user_id2idx[raw_u], item_id2idx[raw_i]] = True
            except (FileNotFoundError, AssertionError) as e:
                print(f"Warning: {filename} skipped or failed. {e}")
            return mask

        train_mask = build_dense_mask("train.tsv")
        test_mask = build_dense_mask("test.tsv")
        validation_mask = build_dense_mask("valid.tsv")

    return user_id2idx, item_id2idx, predicted_scores, train_mask, test_mask, validation_mask


def build_wide_evaluation_dataframe_v2(
    model_names: list[str],
    loader_func: Callable,
    user_info: dict[int, tuple[int, str, str, str]],
    item_info: dict[int, tuple[str, int, list[str]]],
    ground_truth: dict[tuple[int, int], float],
) -> pd.DataFrame:
    """
    Build one row per (user_id, item_id) pair.

    Columns include:
      - user_id, item_id
      - data_split_type
      - one prediction column per model
      - ground_truth
      - user and item metadata

    Predictions and masks are aligned by raw IDs, not assumed
    to have identical positional ordering across models.
    """
    if not model_names:
        raise ValueError("model_names must contain at least one model.")

    df = None
    base_users = None
    base_items = None
    base_split = None

    for model_name in model_names:
        filename = f"{model_name}.npz"

        user_id2idx, item_id2idx, predicted_scores, train_mask, test_mask, validation_mask = loader_func(filename)

        predicted_scores = np.asarray(predicted_scores)
        train_mask = np.asarray(train_mask, dtype=bool)
        test_mask = np.asarray(test_mask, dtype=bool)
        validation_mask = np.asarray(validation_mask, dtype=bool)

        num_users, num_items = predicted_scores.shape
        expected_shape = (num_users, num_items)
        if train_mask.shape != expected_shape or test_mask.shape != expected_shape or validation_mask.shape != expected_shape:
            raise ValueError(f"{model_name}: prediction and mask shapes do not match.")

        # Recover raw IDs in their actual matrix-index order.
        def ids_in_index_order(id2idx, expected_length, name):
            ids = np.empty(expected_length, dtype=np.int64)
            seen = np.zeros(expected_length, dtype=bool)

            for raw_id, idx in id2idx.items():
                idx = int(idx)
                if idx < 0 or idx >= expected_length or seen[idx]:
                    raise ValueError(f"{model_name}: invalid {name} index mapping.")  # noqa: B023
                ids[idx] = int(raw_id)
                seen[idx] = True

            if not seen.all() or len(id2idx) != expected_length:
                raise ValueError(f"{model_name}: {name} mapping is incomplete.")  # noqa: B023

            return ids

        users = ids_in_index_order(user_id2idx, num_users, "user")
        items = ids_in_index_order(item_id2idx, num_items, "item")

        # A pair must not belong to multiple splits.
        overlap = (train_mask & test_mask) | (train_mask & validation_mask) | (test_mask & validation_mask)
        if overlap.any():
            raise ValueError(f"{model_name}: train/test/validation masks overlap for {int(overlap.sum())} user-item pairs.")

        if df is None:
            # Construct the master grid in the first model's ID order.
            base_users = users.copy()
            base_items = items.copy()

            u_idx, i_idx = np.indices(expected_shape)

            df = pd.DataFrame({"user_id": users[u_idx.ravel()], "item_id": items[i_idx.ravel()]})
            split = np.full(expected_shape, "unassigned", dtype=object)
            split[train_mask] = "train"
            split[test_mask] = "test"
            split[validation_mask] = "validation"
            base_split = split.ravel()
            df["data_split_type"] = base_split

        else:
            # Verify that all models cover the same ID sets.
            assert base_users is not None
            assert base_items is not None
            if set(users) != set(base_users) or set(items) != set(base_items):
                raise ValueError(f"{model_name}: user/item ID sets differ from the first model.")

            # Convert the master grid's ID order to this model's
            # positional order before flattening predictions.
            user_positions = np.array([user_id2idx[int(uid)] for uid in base_users])
            item_positions = np.array([item_id2idx[int(iid)] for iid in base_items])

            predicted_scores = predicted_scores[np.ix_(user_positions, item_positions)]

            current_split = np.full(expected_shape, "unassigned", dtype=object)
            current_split[train_mask] = "train"
            current_split[test_mask] = "test"
            current_split[validation_mask] = "validation"

            current_split = current_split[np.ix_(user_positions, item_positions)].ravel()

            assert base_split is not None
            if not np.array_equal(current_split, base_split):
                raise ValueError(f"{model_name}: split assignments differ from the first model after ID alignment.")

        # Predictions now follow the master dataframe's row order.
        df[model_name] = predicted_scores.ravel().astype(np.float32)

    assert df is not None

    # Map ground truth using the exact (user_id, item_id) pair.
    gt = pd.Series(ground_truth, name="ground_truth", dtype=np.float32)
    pair_index = pd.MultiIndex.from_arrays([df["user_id"], df["item_id"]])
    df["ground_truth"] = gt.reindex(pair_index).to_numpy()

    # Join user metadata.
    user_meta = pd.DataFrame.from_dict(
        user_info,
        orient="index",
        columns=["age", "gender", "occupation", "zip_code"],
    )
    user_meta.index = user_meta.index.astype("int64")
    user_meta.index.name = "user_id"

    # Join item metadata.
    item_meta = pd.DataFrame.from_dict(
        item_info,
        orient="index",
        columns=["title", "year", "genres"],
    )
    item_meta.index = item_meta.index.astype("int64")
    item_meta.index.name = "item_id"

    df = df.merge(user_meta.reset_index(), on="user_id", how="left", validate="many_to_one")
    df = df.merge(item_meta.reset_index(), on="item_id", how="left", validate="many_to_one")

    # Add age groups
    df["age_group"] = pd.cut(
        df["age"],
        bins=[0, 18, 26, 36, 46, 56, 101],
        labels=["<18", "18-25", "26-35", "36-45", "46-55", "56+"],
        right=False,
    )

    for col in ["gender", "occupation", "age_group", "data_split_type"]:
        df[col] = df[col].astype("category")

    # Final integrity checks.
    if df.duplicated(["user_id", "item_id"]).any():
        raise AssertionError("Duplicate user-item pairs found.")

    assert base_users is not None
    assert base_items is not None
    expected_rows = len(base_users) * len(base_items)
    if len(df) != expected_rows:
        raise AssertionError(f"Expected {expected_rows} rows, got {len(df)}.")

    return df

In [4]:
df = (
    build_wide_evaluation_dataframe_v2(
        model_names=MODELS,
        loader_func=load_evaluation_data,
        user_info=RecommenderDataLoader().load_user_info(file_path=Path("..", "..", "data", "ml-100k", "ml-100k.user")),
        item_info=RecommenderDataLoader().load_item_info(file_path=Path("..", "..", "data", "ml-100k", "ml-100k.item")),
        ground_truth=RecommenderDataLoader().load_interaction_ratings(file_path=Path("..", "..", "data", "ml-100k", "ml-100k.inter")),
    )
)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1586126 entries, 0 to 1586125
Data columns (total 30 columns):
 #   Column             Non-Null Count    Dtype   
---  ------             --------------    -----   
 0   user_id            1586126 non-null  int64   
 1   item_id            1586126 non-null  int64   
 2   data_split_type    1586126 non-null  category
 3   BPR                1586126 non-null  float32 
 4   EASE               1586126 non-null  float32 
 5   FISM               1586126 non-null  float32 
 6   ItemKNN            1586126 non-null  float32 
 7   UserKNN            1586126 non-null  float32 
 8   SLIMElastic        1586126 non-null  float32 
 9   NeuMF              1586126 non-null  float32 
 10  LightGCN           1586126 non-null  float32 
 11  NGCF               1586126 non-null  float32 
 12  Pop                1586126 non-null  float32 
 13  Random             1586126 non-null  float32 
 14  TunedAugmentation  1586126 non-null  float32 
 15  TunedCascade   

In [5]:
display(df.groupby("user_id").size().describe())

count     943.0
mean     1682.0
std         0.0
min      1682.0
25%      1682.0
50%      1682.0
75%      1682.0
max      1682.0
dtype: float64

In [6]:
# dimension -> {group label -> boolean mask over matrix rows (users) or columns (items)}
GroupMasks = dict[str, dict[str, np.ndarray]]


@dataclass(frozen=True)
class EvaluationData:
    scores: dict[str, np.ndarray]  # model -> (n_users, n_items)
    hidden: np.ndarray  # (n_users, n_items) bool
    relevant: np.ndarray  # (n_users, n_items) bool
    users: pd.DataFrame  # attributes, row-aligned with the matrices
    items: pd.DataFrame  # attributes, column-aligned with the matrices


def build_evaluation_data(df: pd.DataFrame, models: list[str]) -> EvaluationData:
    """Turn the long (user, item) grid dataframe into aligned matrices."""
    df = df.sort_values(["user_id", "item_id"]).reset_index(drop=True)
    user_ids = np.sort(df["user_id"].unique())
    item_ids = np.sort(df["item_id"].unique())
    shape = (len(user_ids), len(item_ids))
    if len(df) != shape[0] * shape[1]:
        raise ValueError("df is not the full user x item grid.")

    split = df["data_split_type"].astype(str).to_numpy().reshape(shape)
    return EvaluationData(
        scores={m: df[m].to_numpy().reshape(shape) for m in models},
        hidden=np.isin(split, HIDDEN_SPLITS),
        relevant=split == TEST_SPLIT,
        users=df.drop_duplicates("user_id").set_index("user_id").loc[user_ids].copy(),
        items=df.drop_duplicates("item_id").set_index("item_id").loc[item_ids].copy(),
    )


def quartile_labels(values: np.ndarray, labels: list[str]) -> pd.Categorical:
    """Equal-sized buckets; ranking first so ties cannot break qcut."""
    ranks = pd.Series(values).rank(method="first").to_numpy()
    return pd.qcut(ranks, q=len(labels), labels=labels)


def masks_by_category(series: pd.Series) -> dict[str, np.ndarray]:
    """One boolean mask per group, in category order (or sorted order)."""
    if isinstance(series.dtype, pd.CategoricalDtype):
        labels = series.cat.categories
    else:
        labels = sorted(series.dropna().unique())
    values = series.to_numpy()
    return {str(label): values == label for label in labels}


def make_user_groups(data: EvaluationData) -> GroupMasks:
    users = data.users.copy()
    users["activity"] = quartile_labels(data.hidden.sum(axis=1), ACTIVITY_LABELS)
    return {dim: masks_by_category(users[dim]) for dim in USER_DIMS}


def make_item_groups(data: EvaluationData) -> GroupMasks:
    items = data.items.copy()  # `genres` must be a list of strings per item
    items["decade"] = (items["year"] // 10 * 10).astype(str) + "s"
    items["popularity_bucket"] = quartile_labels(data.hidden.sum(axis=0), POPULARITY_LABELS)

    all_genres = sorted({genre for genres in items["genres"] for genre in genres})
    genre_masks = {g: items["genres"].apply(lambda genres, g=g: g in genres).to_numpy() for g in all_genres}

    return {
        "decade": masks_by_category(items["decade"]),
        "popularity_bucket": masks_by_category(items["popularity_bucket"]),
        "genre": genre_masks,
    }


def evaluate(
    data: EvaluationData,
    model: str,
    *,
    user_mask: np.ndarray | None = None,
    item_mask: np.ndarray | None = None,
    distance_matrix: np.ndarray | None = None,
    k: int = K,
) -> dict[str, float]:
    """
    Metrics for one model on a user and/or item group.

    Item groups only restrict which test items count as relevant; the ranking
    still runs over the full catalog. Beyond-accuracy metrics are computed
    only for user groups, and only if a distance matrix is passed.
    """
    scores, hidden, relevant = data.scores[model], data.hidden, data.relevant
    if item_mask is not None:
        relevant = relevant & item_mask[None, :]
    if user_mask is not None:
        scores, hidden, relevant = scores[user_mask], hidden[user_mask], relevant[user_mask]
    if not relevant.any():
        return {}

    metrics = RankingMetrics.calculate_all_metrics(scores, hidden, relevant, k)
    if distance_matrix is not None and item_mask is None:
        metrics |= BeyondAccuracyMetrics.calculate_all_metrics(scores, hidden, relevant, distance_matrix, k)
    return metrics


def user_group_table(data: EvaluationData, user_groups: GroupMasks, distance_matrix: np.ndarray, k: int = K) -> pd.DataFrame:
    rows = [
        {
            "model": model,
            "dim": dim,
            "group": group,
            "n_users": int(mask.sum()),
            **evaluate(data, model, user_mask=mask, distance_matrix=distance_matrix, k=k),
        }
        for model in data.scores
        for dim, groups in user_groups.items()
        for group, mask in groups.items()
    ]
    return pd.DataFrame(rows)


def item_group_table(data: EvaluationData, item_groups: GroupMasks, k: int = K) -> pd.DataFrame:
    rows = [
        {"model": model, "dim": dim, "group": group, "n_test": int((data.relevant & mask).sum()), **evaluate(data, model, item_mask=mask, k=k)}
        for model in data.scores
        for dim, groups in item_groups.items()
        for group, mask in groups.items()
    ]
    return pd.DataFrame(rows)

In [7]:
def per_user_metric(data: EvaluationData, model: str, metric: str, item_mask: np.ndarray, k: int = K) -> np.ndarray:
    relevant = data.relevant & item_mask[None, :]
    return RankingMetrics.calculate_per_user_metrics(data.scores[model], data.hidden, relevant, k)[metric]


def _flatten(groups: GroupMasks) -> list[tuple[str, str, np.ndarray]]:
    return [(dim, group, mask) for dim, by_group in groups.items() for group, mask in by_group.items()]


def screen_cells(
    data: EvaluationData,
    user_groups: GroupMasks,
    item_groups: GroupMasks,
    models: list[str],
    metric: str = f"recall@{K}",
    min_users: int = 20,
    min_test: int = 30,
    k: int = K,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Evaluate every (user group, item group) cell, including the 'all' marginals.

    Returns
    -------
    cells   : one row per (cell, model). `lift` = the model's per-user metric minus
              the mean of the other models, tested with a paired t-test.
    winners : one row per cell: best model, runner-up, and whether the margin is significant.
    """
    if len(models) < 2:
        raise ValueError("Need at least two models.")
    n_users, n_items = data.relevant.shape
    user_groups = {"all": {"all": np.ones(n_users, dtype=bool)}, **user_groups}
    item_groups = {"all": {"all": np.ones(n_items, dtype=bool)}, **item_groups}

    # Rank once per model, then compute per-user metrics once per (item group).
    top_k = {m: RankingMetrics.rank_top_k(data.scores[m], data.hidden, k) for m in models}
    per_user: dict[tuple, np.ndarray] = {}  # (item_dim, item_group) -> (n_models, n_users)
    test_counts: dict[tuple, np.ndarray] = {}
    for idim, ig, imask in _flatten(item_groups):
        relevant = data.relevant & imask[None, :]
        test_counts[idim, ig] = relevant.sum(axis=1)
        per_user[idim, ig] = np.stack([
            RankingMetrics.calculate_per_user_metrics(data.scores[m], data.hidden, relevant, k, ranked=top_k[m])[metric]
            for m in models
        ])

    cell_rows, winner_rows = [], []
    for (udim, ug, umask), (idim, ig, _) in product(_flatten(user_groups), _flatten(item_groups)):
        values = per_user[idim, ig][:, umask]
        values = values[:, ~np.isnan(values[0])]  # users with >= 1 test item in this cell
        n, n_test = values.shape[1], int(test_counts[idim, ig][umask].sum())
        if n < min_users or n_test < min_test:
            continue
        cell_key = {"user_dim": udim, "user_group": ug, "item_dim": idim, "item_group": ig, "n_users": n, "n_test": n_test}

        # Lift: each model minus the mean of the others, per user.
        lift = values - (values.sum(axis=0) - values) / (len(models) - 1)
        se = lift.std(axis=1, ddof=1) / np.sqrt(n)
        t = np.divide(lift.mean(axis=1), se, out=np.zeros(len(models)), where=se > 0)
        p = 2 * stats.t.sf(np.abs(t), df=n - 1)
        for j, model in enumerate(models):
            cell_rows.append({**cell_key, "model": model, "score": values[j].mean(),
                              "lift": lift[j].mean(), "se": se[j], "p": p[j]})

        # Winner: best mean score, tested against the runner-up.
        means = values.mean(axis=1)
        best, second = np.argsort(-means)[:2]
        diff = values[best] - values[second]
        w_se = diff.std(ddof=1) / np.sqrt(n)
        w_t = diff.mean() / w_se if w_se > 0 else 0.0
        winner_rows.append({**cell_key, "best": models[best], "runner_up": models[second],
                            "margin": diff.mean(), "p": 2 * stats.t.sf(abs(w_t), df=n - 1)})

    cells, winners = pd.DataFrame(cell_rows), pd.DataFrame(winner_rows)
    cells["lift_q"] = stats.false_discovery_control(cells["p"].to_numpy())      # Benjamini-Hochberg
    winners["margin_q"] = stats.false_discovery_control(winners["p"].to_numpy())
    return cells, winners


def add_interaction(cells: pd.DataFrame) -> pd.DataFrame:
    """Cross cells only, with: lift beyond what the row and column marginals predict (per model)."""
    ua, ia = cells["user_dim"].eq("all"), cells["item_dim"].eq("all")
    user_lift = cells.loc[ia, ["model", "user_dim", "user_group", "lift"]].rename(columns={"lift": "user_lift"})
    item_lift = cells.loc[ua, ["model", "item_dim", "item_group", "lift"]].rename(columns={"lift": "item_lift"})
    overall = cells.loc[ua & ia, ["model", "lift"]].rename(columns={"lift": "overall_lift"})

    cross = (
        cells[~ua & ~ia]
        .merge(user_lift, on=["model", "user_dim", "user_group"])
        .merge(item_lift, on=["model", "item_dim", "item_group"])
        .merge(overall, on="model")
    )
    cross["expected_lift"] = cross["user_lift"] + cross["item_lift"] - cross["overall_lift"]
    cross["interaction"] = cross["lift"] - cross["expected_lift"]

    # Approximate: treats the expected lift as fixed. Good enough for screening.
    z = (cross["interaction"] / cross["se"].replace(0, np.nan)).fillna(0)
    cross["interaction_q"] = stats.false_discovery_control(2 * stats.norm.sf(z.abs()))
    return cross


def rank_dimension_pairs(cross: pd.DataFrame, winners: pd.DataFrame, alpha: float = ALPHA) -> pd.DataFrame:
    """
    Which (user dim x item dim) matrices are worth plotting?

    n_distinct_winners : how many different models win at least one cell (1 = one model wins everywhere, boring)
    share_decisive     : share of cells where the winner is significantly ahead of the runner-up
    share_interacting  : share of (cell, model) pairs that deviate from what the marginals predict
    """
    keys = ["user_dim", "item_dim"]
    interaction = (
        cross.assign(interacting=cross["interaction_q"] < alpha, abs_interaction=cross["interaction"].abs())
        .groupby(keys)
        .agg(share_interacting=("interacting", "mean"), mean_abs_interaction=("abs_interaction", "mean"))
    )
    cross_winners = winners[(winners["user_dim"] != "all") & (winners["item_dim"] != "all")]
    wins = (
        cross_winners.assign(decisive=cross_winners["margin_q"] < alpha)
        .groupby(keys)
        .agg(n_cells=("best", "size"), n_distinct_winners=("best", "nunique"), share_decisive=("decisive", "mean"))
    )
    return wins.join(interaction).sort_values(["n_distinct_winners", "share_interacting"], ascending=False)


def notable_cells(cross: pd.DataFrame, by: str = "lift", model: str | None = None,
                  alpha: float = ALPHA, n: int = 15) -> pd.DataFrame:
    """Largest significant cells, ranked by |lift| or |interaction|. `model` filters to one model."""
    frame = cross if model is None else cross[cross["model"] == model]
    return frame[frame[f"{by}_q"] < alpha].sort_values(by, key=np.abs, ascending=False).head(n)

In [ ]:
def _star_labels(values: pd.DataFrame, q: pd.DataFrame, alpha: float, show_values: bool) -> pd.DataFrame:
    """Annotation text: the value (optional) plus * where FDR q < alpha. Empty for missing cells."""
    stars = pd.DataFrame(np.where(q < alpha, "*", ""), index=values.index, columns=values.columns)
    text = values.round(2).astype(str) + stars if show_values else stars
    return text.where(values.notna(), "")


def plot_model_profile(
    cells: pd.DataFrame,
    models: list[str],
    side: str = "user",            # "user" -> models x user groups, "item" -> models x item groups
    value: str = "lift",           # "lift" (relative to other models) or "score" (absolute metric)
    metric: str = f"recall@{K}",
    alpha: float = ALPHA,
    output_dir: str | Path = Path("..", "..", "figures")
) -> Path:
    """Models x groups heatmap for one side: where is each model strong or weak?"""
    dim_col, group_col, other_dim = ("user_dim", "user_group", "item_dim") if side == "user" else ("item_dim", "item_group", "user_dim")
    marginals = cells[(cells[other_dim] == "all") & (cells[dim_col] != "all")]
    keys = list(marginals[[dim_col, group_col]].drop_duplicates().itertuples(index=False, name=None))

    def pivot(column: str) -> pd.DataFrame:
        return marginals.pivot(index="model", columns=[dim_col, group_col], values=column).reindex(
            index=models, columns=pd.MultiIndex.from_tuples(keys)
        )

    values, q = pivot(value), pivot("lift_q")
    fig, ax = plt.subplots(figsize=(max(12, 0.6 * len(keys)), 0.6 * len(models) + 3), constrained_layout=True)
    if value == "lift":
        limit = float(np.nanquantile(np.abs(values.to_numpy(dtype=float)), 0.98))
        limit = limit or 1.0
        kwargs = {"cmap": "RdBu", "vmin": -limit, "vmax": limit}
        cbar_label = f"{metric}: model minus mean of the other models"
    else:
        kwargs = {"cmap": "viridis"}
    cbar_label = metric
    sns.heatmap(values, annot=_star_labels(values, q, alpha, True), fmt="", annot_kws={"size": 7},
                xticklabels=[g for _, g in keys], linewidths=0.5, linecolor="white",
                cbar_kws={"label": cbar_label, "shrink": 0.7}, ax=ax, **kwargs)

    # Dimension names above the columns, thick separators between dimensions
    start = 0
    for dim, grp in groupby(keys, key=lambda key: key[0]):
        width = len(list(grp))
        ax.text(start + width / 2, 1.01, dim, transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontweight="bold")
        if start:
            ax.axvline(start, color="black", lw=2)
        start += width

    noun = "user groups" if side == "user" else "item groups"
    ax.set_title(
        f"Where is each model strong? {metric} per {noun}\n"
        + ("Blue = better than the average of the other models, red = worse. " if value == "lift" else "")
        + f"* = significant (FDR q < {alpha})",
        pad=30,
    )
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.setp(ax.get_xticklabels(), rotation=60, ha="right")

    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    filename = f"model_profile_{side}_{value}_{_sanitize_filename(metric)}.png"
    filepath = out_path / filename

    plt.savefig(filepath, bbox_inches="tight", dpi=300)
    plt.close(fig)  # Prevents memory accumulation in loops
    return filepath

def _sanitize_filename(name: str) -> str:
    """Helper to convert strings into clean, cross-platform filename paths."""
    clean = name.replace("@", "_at_").replace(" ", "_").lower()
    return re.sub(r"[^\w\-]", "", clean)

def plot_pair(
    cross: pd.DataFrame,
    winners: pd.DataFrame,
    user_dim: str,
    item_dim: str,
    user_groups: GroupMasks,
    item_groups: GroupMasks,
    models: list[str],
    metric: str = f"recall@{K}",
    alpha: float = ALPHA,
    max_annotated_cells: int = 6000,
    output_dir: str | Path = Path("..", "..", "figures")
) -> Path:
    """Winner map + one lift panel per model for a (user dim x item dim) matrix."""
    cells = cross[(cross["user_dim"] == user_dim) & (cross["item_dim"] == item_dim)]
    won = winners[(winners["user_dim"] == user_dim) & (winners["item_dim"] == item_dim)]
    if cells.empty:
        raise ValueError(f"No cells for {user_dim} x {item_dim} (all below the size thresholds?).")

    rows = [g for g in user_groups[user_dim] if g in set(cells["user_group"])]
    cols = [g for g in item_groups[item_dim] if g in set(cells["item_group"])]
    row_labels = [f"{g} (n={int(user_groups[user_dim][g].sum())} users)" for g in rows]
    col_labels = [f"{g} ({int(item_groups[item_dim][g].sum())} items)" for g in cols]
    annotate_values = len(rows) * len(cols) <= max_annotated_cells  # big matrices: colours and stars only

    def grid(frame: pd.DataFrame, column: str) -> pd.DataFrame:
        return frame.pivot(index="user_group", columns="item_group", values=column).reindex(index=rows, columns=cols)

    limit = cells["lift"].abs().quantile(0.98) or 1.0
    n_panels = len(models) + 1
    ncols = min(5, n_panels)
    nrows = math.ceil(n_panels / ncols)

    # Dynamically scale column width based on the number of item groups
    panel_width = max(5.0, 0.35 * len(cols))
    panel_height = max(3.0, 0.4 * len(rows) + 2.5)

    # Dynamically shrink font size as the matrix grows (bounds between 4pt and 7pt)
    cell_fontsize = max(4, min(7, int(60 / max(len(rows), len(cols)))))
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_width * ncols, nrows * panel_height),
                             constrained_layout=True, squeeze=False)
    flat = axes.ravel()
    for ax in flat[n_panels:]:
        ax.set_visible(False)

    # Panel 0: winner map
    model_idx = {m: i for i, m in enumerate(models)}
    best = grid(won, "best")
    codes = best.apply(lambda column: column.map(model_idx)).astype(float)
    decisive = grid(won, "margin_q") < alpha
    labels = pd.DataFrame(np.where(decisive, "*", ""), index=best.index, columns=best.columns)
    sns.heatmap(codes, cmap=ListedColormap([MODEL_COLORS[m] for m in models]), vmin=-0.5, vmax=len(models) - 0.5,
                annot=labels, fmt="", annot_kws={"size": cell_fontsize, "color": "white"}, cbar=False,
                xticklabels=col_labels, yticklabels=row_labels, linewidths=0.5, linecolor="white", ax=flat[0])
    flat[0].set_facecolor("lightgrey")
    flat[0].set_title("Best model per cell\n(* = significantly ahead of the runner-up)", fontsize=10)

    # Panels 1..n: each model relative to the others
    for j, (ax, model) in enumerate(zip(flat[1:n_panels], models, strict=True), start=1):
        frame = cells[cells["model"] == model]
        lift, q = grid(frame, "lift"), grid(frame, "lift_q")
        sns.heatmap(lift, cmap="RdBu", vmin=-limit, vmax=limit, annot=_star_labels(lift, q, alpha, annotate_values),
                    fmt="", annot_kws={"size": cell_fontsize}, cbar=False, xticklabels=col_labels,
                    yticklabels=row_labels if j % ncols == 0 else False, linewidths=0.5, linecolor="white", ax=ax)
        ax.set_facecolor("lightgrey")
        ax.set_title(f"{model}\nvs. mean of other models", fontsize=10, color=MODEL_COLORS[model], fontweight="bold")
    for ax in flat[:n_panels]:
        ax.set_xlabel(item_dim)
        ax.set_ylabel(user_dim)
        plt.setp(ax.get_xticklabels(), rotation=60, ha="right", fontsize=8)

    fig.colorbar(ScalarMappable(Normalize(-limit, limit), cmap="RdBu"), ax=list(flat[:n_panels]),
                 location="bottom", shrink=0.4, label=f"{metric}: model minus mean of the other models (blue = better)")
    fig.legend(handles=[Patch(color=MODEL_COLORS[m], label=m) for m in models if m in set(won["best"])],
               title="Best model", loc="outside right center")
    fig.suptitle(
        f"{user_dim} × {item_dim}: {metric}\n"
        f"Rows = user groups, columns = item groups (cells = test items of that item group for users of that user group). "
        f"* = significant (FDR q < {alpha}). Grey = too few users or test items.",
        fontsize=11,
    )

    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    filename = f"pair_{_sanitize_filename(user_dim)}_x_{_sanitize_filename(item_dim)}_{_sanitize_filename(metric)}.png"
    filepath = out_path / filename

    plt.savefig(filepath, bbox_inches="tight", dpi=400)
    plt.close(fig)  # Prevents memory accumulation in loops
    return filepath

In [35]:
data = build_evaluation_data(df, MODELS)
user_groups, item_groups = make_user_groups(data), make_item_groups(data)

cells, winners = screen_cells(data, user_groups, item_groups, MODELS)
cross = add_interaction(cells)

# 1. Where is each model strong? (the headline plots for the task)
plot_model_profile(cells, MODELS, side="user")
plot_model_profile(cells, MODELS, side="item")

# 2. Which dimension combinations deserve a closer look?
pairs = rank_dimension_pairs(cross, winners)
print(pairs.head(10))

# 3. Plot the top combinations
for user_dim, item_dim in pairs.head(3).index:
    plot_pair(cross, winners, user_dim, item_dim, user_groups, item_groups, MODELS)

                              n_cells  n_distinct_winners  share_decisive  \
user_dim   item_dim                                                         
activity   genre                   59                  16             0.0   
occupation genre                  123                  16             0.0   
age_group  decade                  30                  14             0.0   
           genre                   75                  14             0.0   
occupation decade                  39                  13             0.0   
gender     genre                   35                  13             0.0   
occupation popularity_bucket       28                  12             0.0   
activity   decade                  19                  10             0.0   
gender     decade                  14                  10             0.0   
age_group  popularity_bucket       18                   9             0.0   

                              share_interacting  mean_abs_interaction  
use

In [22]:
# 4. Concrete findings as tables
notable_cells(cross, by="lift")                      # biggest significant wins/losses

,user_dim,user_group,item_dim,item_group,n_users,n_test,model,score,lift,se,p,lift_q,user_lift,item_lift,overall_lift,expected_lift,interaction,interaction_q
6813,activity,low,genre,Horror,27,30,Pop,0.000000,-0.515251,0.066741,3.425867e-08,4.595918e-07,-0.257561,-0.157582,-0.170450,-0.244693,-0.270558,0.002128
6814,activity,low,genre,Horror,27,30,Random,0.000000,-0.515251,0.066741,3.425867e-08,4.595918e-07,-0.343553,-0.203438,-0.252472,-0.294519,-0.220732,0.032098
6904,activity,low,genre,War,51,55,Random,0.000000,-0.502307,0.056487,7.155609e-12,1.497348e-10,-0.343553,-0.358155,-0.252472,-0.449236,-0.053071,1.000000
6724,activity,low,genre,Action,103,129,Random,0.000000,-0.460879,0.037507,7.688291e-22,3.655204e-20,-0.343553,-0.332329,-0.252472,-0.423410,-0.037470,1.000000
6202,occupation,student,genre,War,98,173,Random,0.005102,-0.453713,0.040778,5.061834e-19,1.988271e-17,-0.260735,-0.358155,-0.252472,-0.366418,-0.087295,0.531497
6940,activity,mid-low,decade,1970s,46,52,Random,0.000000,-0.451407,0.059334,1.290636e-09,2.034539e-08,-0.284700,-0.330647,-0.252472,-0.362875,-0.088531,0.949440
7228,activity,mid-low,genre,Sci-Fi,104,140,Random,0.009615,-0.440328,0.039107,1.185261e-19,4.905077e-18,-0.284700,-0.375607,-0.252472,-0.407836,-0.032492,1.000000
4852,occupation,marketing,popularity_bucket,head,26,150,Random,0.000000,-0.439293,0.061280,1.634936e-07,1.975992e-06,-0.352872,-0.331460,-0.252472,-0.431860,-0.007433,1.000000
6868,activity,low,genre,Sci-Fi,55,60,Random,0.000000,-0.438503,0.055712,1.594518e-10,2.909681e-09,-0.343553,-0.375607,-0.252472,-0.466688,0.028186,1.000000
6886,activity,low,genre,Thriller,121,153,Random,0.000000,-0.437368,0.034585,8.108174e-24,4.270612e-22,-0.343553,-0.290211,-0.252472,-0.381292,-0.056076,0.880230


In [23]:
notable_cells(cross, by="interaction", model="EASE") # surprising cells for one model

,user_dim,user_group,item_dim,item_group,n_users,n_test,model,score,lift,se,p,lift_q,user_lift,item_lift,overall_lift,expected_lift,interaction,interaction_q
4645,occupation,librarian,popularity_bucket,high-mid,35,128,EASE,0.009524,-0.002033,0.003606,0.576637,0.718471,0.080391,-0.001920,0.039566,0.038904,-0.040937,7.188953e-28
7903,activity,high,popularity_bucket,low-mid,180,486,EASE,0.000000,-0.000563,0.000254,0.027944,0.082470,0.022084,-0.003352,0.039566,-0.020834,0.020271,0.000000e+00
1711,age_group,46-55,popularity_bucket,high-mid,82,232,EASE,0.004065,-0.007807,0.003407,0.024515,0.074901,0.052778,-0.001920,0.039566,0.011292,-0.019099,1.155112e-06
6445,occupation,writer,popularity_bucket,high-mid,32,122,EASE,0.006250,-0.011598,0.005540,0.044576,0.117851,0.047583,-0.001920,0.039566,0.006097,-0.017695,4.608870e-02
7885,activity,high,popularity_bucket,long-tail,51,104,EASE,0.000000,-0.000577,0.000577,0.322126,0.482174,0.022084,-0.000350,0.039566,-0.017832,0.017256,2.107497e-194
8065,activity,high,genre,Documentary,49,58,EASE,0.000000,-0.003601,0.001846,0.056915,0.141942,0.022084,-0.003180,0.039566,-0.020662,0.017060,2.007277e-18
2035,age_group,56+,popularity_bucket,high-mid,32,101,EASE,0.015625,0.001955,0.003758,0.606615,0.745205,0.057916,-0.001920,0.039566,0.016430,-0.014475,4.645149e-03
7921,activity,high,popularity_bucket,high-mid,233,1347,EASE,0.002160,-0.006086,0.001290,0.000004,0.000038,0.022084,-0.001920,0.039566,-0.019402,0.013316,4.986321e-23
253,age_group,18-25,popularity_bucket,low-mid,85,194,EASE,0.000000,-0.000231,0.000231,0.320183,0.482174,0.031866,-0.003352,0.039566,-0.011052,0.010821,0.000000e+00
1693,age_group,46-55,popularity_bucket,low-mid,34,79,EASE,0.000000,-0.000124,0.000124,0.324587,0.482194,0.052778,-0.003352,0.039566,0.009860,-0.009984,0.000000e+00


In [24]:
notable_cells(cross, by="interaction") 

,user_dim,user_group,item_dim,item_group,n_users,n_test,model,score,lift,se,p,lift_q,user_lift,item_lift,overall_lift,expected_lift,interaction,interaction_q
6813,activity,low,genre,Horror,27,30,Pop,0.000000,-0.515251,0.066741,3.425867e-08,4.595918e-07,-0.257561,-0.157582,-0.170450,-0.244693,-0.270558,2.128272e-03
6814,activity,low,genre,Horror,27,30,Random,0.000000,-0.515251,0.066741,3.425867e-08,4.595918e-07,-0.343553,-0.203438,-0.252472,-0.294519,-0.220732,3.209766e-02
6808,activity,low,genre,Horror,27,30,UserKNN,0.703704,0.229847,0.053192,2.015440e-04,1.309259e-03,0.033317,0.028713,0.009462,0.052567,0.177280,2.978191e-02
7858,activity,high,decade,1980s,219,783,Random,0.002892,-0.322742,0.016429,3.938823e-50,6.761646e-48,-0.167000,-0.279944,-0.252472,-0.194472,-0.128269,4.318769e-13
7857,activity,high,decade,1980s,219,783,Pop,0.117939,-0.200927,0.015757,3.428693e-28,2.176985e-26,-0.085491,-0.159455,-0.170450,-0.074496,-0.126431,7.940448e-14
4937,occupation,other,decade,1970s,34,55,SLIMElastic,0.306373,-0.042215,0.030748,1.790316e-01,3.221855e-01,0.042652,0.053117,0.035445,0.060324,-0.102539,2.969203e-02
7768,activity,high,decade,1930s,76,108,Random,0.000000,-0.100684,0.022376,2.444695e-05,1.943596e-04,-0.167000,-0.088042,-0.252472,-0.002571,-0.098113,5.224816e-04
7767,activity,high,decade,1930s,76,108,Pop,0.000000,-0.100684,0.022376,2.444695e-05,1.943596e-04,-0.085491,-0.088042,-0.170450,-0.003083,-0.097600,5.771839e-04
8164,activity,high,genre,Musical,165,327,Random,0.000000,-0.134949,0.015296,1.597987e-15,4.658283e-14,-0.167000,-0.126579,-0.252472,-0.041107,-0.093842,5.111363e-08
7839,activity,high,decade,1970s,187,419,Pop,0.171900,-0.106989,0.021608,1.646364e-06,1.646364e-05,-0.085491,-0.098784,-0.170450,-0.013825,-0.093164,7.100669e-04
